# 02 · Model construction

Launches the model-building step. The MIL model is assembled from a **template**
(`bag_attention`) that wires three components:

1. **Embedder** — embeds each instance (conformer) descriptor vector.
2. **Aggregator** — attention pooling over instances into one bag representation.
3. **Predictor** — MLP head producing the endpoint prediction.

We build the model exactly as `ModelTrainer.build_model()` does, injecting the
descriptor count (`input_dim`) discovered during preprocessing.

In [2]:
# --- Bootstrap: make the notebook run from anywhere ---
import os, sys, logging
from pathlib import Path

# Locate the project root (folder that contains the `ppl` package).
here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [here, *here.parents] if (p / 'ppl' / '__init__.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    # Fallback: this notebook lives in <root>/notebooks/
    PROJECT_ROOT = Path('__file__' in globals() and __file__ or '.').resolve().parent.parent

os.chdir(PROJECT_ROOT)                       # pipeline writes outputs relative to cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
print('Project root:', PROJECT_ROOT)

Project root: /Users/vfastovskii/Desktop/milkid


In [3]:
# Path to the experiment YAML. Edit this to point at a different config.
CONFIG_PATH = PROJECT_ROOT / 'ppl/config/experiment_configs/run_config.yaml'
assert CONFIG_PATH.exists(), f'Config not found: {CONFIG_PATH}'
print('Using config:', CONFIG_PATH.relative_to(PROJECT_ROOT))

Using config: ppl/config/experiment_configs/run_config.yaml


## Get `input_dim` from the data

The model needs to know how many descriptors each instance has. We run a light
`setup()` to read it off the data module (same as step 01).

In [4]:
from ppl.config.pipeline_config import PipelineConfig
from ppl.pipeline.config_manager import PipelineConfigManager
from ppl.data.data_loader import MILDataModule

cfg = PipelineConfig.from_yaml(CONFIG_PATH)
cfg_mgr = PipelineConfigManager(cfg)

dm = MILDataModule(cfg_mgr.data_cfg)
dm.setup('fit')
input_dim = len(dm.feature_names)
print('input_dim:', input_dim)

INFO ppl.config.pipeline_config: Using default csv_path: /Users/vfastovskii/Desktop/milkid/ppl/data/trypsin_usrcat.csv
INFO ppl.pipeline.config_override_utils: [EXP CFG] Applied overrides to DataLoaderConfig
INFO ppl.pipeline.config_manager: Using experiment_name 'bace809_cluster_hier_mha_200c_2jul_bs16_test15' for data splits
INFO ppl.pipeline.config_manager: Setting default cache_dir to 'bace809_cluster_hier_mha_200c_2jul_bs16_test15' for data splits
INFO ppl.pipeline.config_override_utils: [EXP CFG] Applied overrides to ModelBuilderConfig
INFO ppl.pipeline.config_override_utils: [EXP CFG] Applied overrides to TrainerOptimConfig
INFO ppl.pipeline.config_override_utils: [EXP CFG] Applied overrides to TrainerConfig
INFO ppl.pipeline.config_manager: Setting global seed to 42
INFO ppl.utils.reproducibility.deterministic_setup: Deterministic setup complete with seed: 42
INFO ppl.utils.reproducibility.deterministic_setup: CUBLAS_WORKSPACE_CONFIG = :4096:8
INFO ppl.utils.reproducibility.det

input_dim: 474


## Build the model

`ModelTrainer` owns model construction in the real pipeline. We instantiate it
from the config sections and call `build_model(input_dim)`.

In [5]:
from ppl.training.model_trainer import ModelTrainer

trainer_obj = ModelTrainer(
    model_cfg=cfg_mgr.model_cfg,
    trainer_cfg=cfg_mgr.trainer_cfg,
    log_save_dir=cfg_mgr.log_save_dir,
    seed=cfg_mgr.seed,
    task=cfg_mgr.task,
    experiment_name=cfg_mgr.trainer_cfg.experiment_name,
)
model = trainer_obj.build_model(input_dim)
print(type(model).__name__, 'built')

INFO ppl.training.model_trainer: [MODEL] Injected input_dim=474 into ModelBuilderConfig
INFO ppl.models.model_builder: [MODEL] Building model from config:
{'active_prototype_kwargs': {'ablation_attention_weight': 0.5,
                             'ablation_candidate_pool': 8,
                             'ablation_impact_weight': 0.5,
                             'ablation_positive_only': True,
                             'active_threshold': 7.0,
                             'candidate_alpha_key': 'alpha',
                             'candidate_selection': 'attention_ablation',
                             'create_sim_threshold': 0.8,
                             'ema_momentum': 0.03,
                             'enabled': True,
                             'max_prototypes': 64,
                             'max_prototypes_per_series': 2,
                             'merge_sim_threshold': 0.9,
                             'min_active_prototypes': 16,
                             'm

MILModelLightningWrapper built


## Inspect the architecture

In [1]:
print(model)

NameError: name 'model' is not defined

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'total params    : {n_params:,}')
print(f'trainable params: {n_train:,}')

## Sanity forward pass

Feed one real batch through the model's `core` to confirm shapes line up
before committing to a full training run.

In [ ]:
import torch

model.eval()
batch = next(iter(dm.train_dataloader()))
bags, y, bag_ids = batch[0], batch[1], batch[2]
padding_mask = batch[3] if len(batch) > 3 else None
cluster_ids = batch[4] if len(batch) > 4 else None
x = bags[0] if isinstance(bags, list) else bags

core_kwargs = {}
if padding_mask is not None:
    core_kwargs['key_padding_mask'] = padding_mask
if cluster_ids is not None:
    core_kwargs['cluster_ids'] = cluster_ids

with torch.no_grad():
    try:
        logit, _ = model.core(x, **core_kwargs)
    except TypeError:
        logit, _ = model.core(x)

print('input  :', tuple(x.shape))
print('output :', tuple(logit.shape))
print('sample predictions:', logit.reshape(-1)[:5].tolist())

---
The architecture is ready. Continue with **`03_model_training.ipynb`** to train it.